# [EDA] 2022년도 전체 시도 세부사업 예산현황 분리·정제 (작업중) — 이슈 #8

> - 입력: `data/raw/칼럼정렬/*2022*.xlsx`의 `정리본_자동`·`Table 1` 시트.
> - QA는 원시값 완전 동일 비교와 소수점 첫째 자리 반올림 후 차이 0.0 비교를 구분한다.
> - 서울 원본행 28의 연속 후보는 반복 머리글을 건너뛴 앞 실제 데이터행인 원본행 24에 병합하고 leaf에서 제외한다.
> - 후보 판정 후 최종 leaf는 8,129행이며, long 변환 시 16,258행이다.
> - LLM 정제 단계는 실행하지 않았으며 이 노트북의 결정론 전처리 범위에 포함하지 않는다.


In [3]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

YEAR = 2022  # 시행계획 연도
PREVIOUS_YEAR = YEAR - 1
CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"

# 경로 설정
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")


def get_sido_dir(sido: str) -> Path:
    """시도별 중간 산출물 디렉터리를 생성하고 경로를 반환한다."""
    sido_dir = INTERIM_DIR / sido
    sido_dir.mkdir(parents=True, exist_ok=True)
    return sido_dir


def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마·공백을 정리하고 '-', '비예산'을 0으로 치환한 뒤 숫자로 변환한다."""
    normalized = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"-": "0", "비예산": "0"})
    )
    return pd.to_numeric(normalized, errors="coerce")


# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

In [4]:
# 칼럼정렬 파일에는 파이프라인 표준 컬럼과 시트가 준비되어 있음.
COLUMN_ALIGNED_DIR = DATA_DIR / "칼럼정렬"
source_file = next(COLUMN_ALIGNED_DIR.glob(f"*{YEAR}*.xlsx"))
df_raw = pd.read_excel(source_file, sheet_name="정리본_자동")

print(df_raw.shape)
df_raw.head(10)

(8472, 12)


,지역,세부사업명,사업분류재정구분,2022년 예산,2021년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,7299445,6946037,353408,5.1,NaN,1,2,4,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),NaN,2991826,3085406,-93580,-3,NaN,1,2,5,데이터행
2,서울,저출생 인식개선 캠페인,공통,58,43,15,34.9,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",1,2,6,데이터행
3,서울,어린이 안전 영상정보 인프라 구축,공통,38039,33280,4759,14.3,지원대상 : 어린이\n지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,1,2,7,데이터행
4,서울,입양아동 가족지원,공통,4780,4120,660,16,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정\n지원내용 : 입양아동양육보조금, 심리치료비 등",1,2,8,데이터행
5,서울,국공립어린이집 등 보육서비스 수준제고,공통,330577,322975,7602,2.4,지원대상 : 국공립어린이집 등\n지원내용 : 보육교직원 인건비 지원,1,2,9,데이터행
6,서울,국공립어린이집 확충,공통,15523,40000,-24477,-61.2,국공립어린이집 이용률 46%달성,1,2,10,데이터행
7,서울,어린이집 이용 영유아 무상보육 확대,공통,791160,854364,-63204,-7.4,지원대상 : 어린이집 이용 영유아\n지원내용 : 해당 연령 보육료,1,2,11,데이터행
8,서울,육아종합지원센터 운영,공통,6293,6104,189,3.1,"지원대상 : 보육교직원 및 영유아 부모\n지원내용 : 보육정보 및 교육, 전문상담 제공",1,2,12,데이터행
9,서울,가정양육수당 지원,공통,115422,171371,-55949,-32.6,지원대상 : 어린이집･유치원 미이용 0~86개월 미만 아동\n지원내용 :10~20만원(월) 양육수당 지원,1,2,13,데이터행


In [5]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

[데이터셋 크기]
 (8472, 12)
[데이터 타입]
 지역             str
세부사업명          str
사업분류재정구분       str
2022년 예산    object
2021년 예산    object
증감액         object
비율          object
주요내용           str
원본표구간        int64
머리글행         int64
원본행          int64
행유형            str
dtype: object
[각 컬럼별 결측치 개수 평균]
 지역          0.000
세부사업명       0.000
사업분류재정구분    0.020
2022년 예산    0.003
2021년 예산    0.005
증감액         0.007
비율          0.000
주요내용        0.021
원본표구간       0.000
머리글행        0.000
원본행         0.000
행유형         0.000
dtype: float64
[중복 행 수] 0
[지역(시도) 목록 확인]
 지역
경기    1348
부산     773
경북     768
경남     733
충남     645
전남     573
강원     542
전북     519
충북     514
인천     398
대전     339
대구     294
울산     270
서울     244
광주     217
제주     174
세종     121
Name: count, dtype: int64


In [6]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

강원 542
경기 1348
경남 733
경북 768
광주 217
대구 294
대전 339
부산 773
서울 244
세종 121
울산 270
인천 398
전남 573
전북 519
제주 174
충남 645
충북 514


In [8]:
# 이전 데이터셋에서 확인한 classify_row 적용
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")
# 2022 페이지 반복헤더/시도경계 제목행
header_noise_pattern = re.compile(
    r"고령사회기본계획|지방자치단체\s*시행계획|세부사업별\s*예산\s*현황"
)


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if header_noise_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_raw[CURRENT_BUDGET_COL] = to_numeric_budget(df_raw[CURRENT_BUDGET_COL])

In [9]:
df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

사업행구분,대분류_소계,세부사업,중분류_소계,헤더반복
지역,,,,
강원,2,521,8,11
경기,2,1311,8,27
경남,2,707,8,16
경북,2,742,8,16
광주,2,204,7,4
대구,2,278,8,6
대전,2,322,8,7
부산,2,748,8,15
서울,2,228,8,6


-> 2022년 문서에서는 모든 시도에 페이지 반복 머리글·시도 경계 제목행이 있어 `헤더반복`이 지역별 3~27건 탐지되었다. 경기가 27건으로 가장 많고, 세종이 3건으로 가장 적으며, 서울은 6건이다.

-> 헤더 반복 및 경계 제목행은 세부사업 분석 대상에서 제외한다.


In [10]:
# 광주 중분류_소계 행 확인
df_gwangju = sido_dfs["광주"]
df_gwangju["사업행구분"] = df_gwangju["세부사업명"].apply(classify_row)
print("[광주] 중분류_소계 행 확인")
print("==================================================")
print(df_gwangju[df_gwangju["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

print("\n")

# 전남도 동일하게 확인
df_jeongnam = sido_dfs["전남"]
df_jeongnam["사업행구분"] = df_jeongnam["세부사업명"].apply(classify_row)
print("[전남] 중분류_소계 행 확인")
print("==================================================")
print(df_jeongnam[df_jeongnam["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

[광주] 중분류_소계 행 확인
1711       1. 함께 일하고 함께 돌보는 사회(공통)
1723         2. 건강하고 능동적인 고령사회(공통)
1735     3. 모두의 역량이 고루 발휘되는 사회(공통)
1746      1. 함께 일하고 함께 돌보는\n사회(자체)
1838      2. 건강하고 능동적인 고령사회 구축(자체)
1877    3. 모두의 역량이 고루\n발휘되는 사회(자체)
1914          4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


[전남] 중분류_소계 행 확인
6226       1. 함께 일하고 함께 돌보는 사회(공통)
6235         2. 건강하고 능동적인 고령사회(공통)
6255         4.인구구조 변화에 대한\n적응(공통)
6270       1. 함께 일하고 함께 돌보는 사회(자체)
6528     2. 건강하고 능동적인\n고령사회 구축(자체)
6645    3. 모두의 역량이 고루\n발휘되는 사회(자체)
6740         4.인구구조 변화에 대한\n적응(자체)
Name: 세부사업명, dtype: str


-> 광주는 `4. 인구구조 변화에 대한 적응(공통)`이 없고,

-> 전남은 `3. 모두의 역량이 고루 발휘되는 사회(공통)`가 없다.

-> 출력된 7개 항목은 모두 정상적인 중분류 형식이므로, 누락된 영역이 원문에도 없는지 `Table 1` 시트와 대조해 원문 구성 차이인지 분류 누락인지를 확인할 필요가 있다.


In [11]:
# Table 1 시트의 시도별 원문 구간에서 누락된 중분류를 대조한다.
df_table1 = pd.read_excel(source_file, sheet_name="Table 1", header=None, dtype="string")
table1_row_text = df_table1.fillna("").agg(" ".join, axis=1).str.replace(r"\s+", "", regex=True)

# `(OO시도·OO교육청)` 형식의 행을 시도 구간 시작점으로 사용한다.
region_header_pattern = re.compile(r"\(([^()･]+)･[^()]*교육청\)")
region_starts = []
for row_idx, row_text in table1_row_text.items():
    match = region_header_pattern.search(row_text)
    if match:
        region_starts.append((row_idx, match.group(1)))

table1_qa_targets = {
    "광주광역시": "4. 인구구조 변화에 대한 적응(공통)",
    "전라남도": "3. 모두의 역량이 고루 발휘되는 사회(공통)",
}

for region_name, target_title in table1_qa_targets.items():
    start_pos = next(i for i, (_, name) in enumerate(region_starts) if name == region_name)
    start_row = region_starts[start_pos][0]
    end_row = (
        region_starts[start_pos + 1][0] if start_pos + 1 < len(region_starts) else len(df_table1)
    )
    region_text = table1_row_text.iloc[start_row:end_row]
    normalized_title = re.sub(r"\s+", "", target_title)
    exact_rows = region_text[region_text.str.contains(re.escape(normalized_title), regex=True)]

    # 번호·공통 표기가 다른 사례도 잡기 위해 핵심 명칭으로 한 번 더 검색한다.
    core_title = re.sub(r"^\d+\.", "", normalized_title).replace("(공통)", "")
    candidate_rows = region_text[region_text.str.contains(re.escape(core_title), regex=True)]

    print(f"[{region_name}] Table 1 행 구간: {start_row + 1}~{end_row}")
    print(f"정확 명칭 일치: {len(exact_rows)}건 / 핵심 명칭 후보: {len(candidate_rows)}건")
    if candidate_rows.empty:
        print(f"원문 구간에서 '{target_title}'을(를) 찾지 못함")
    else:
        display(df_table1.loc[candidate_rows.index].dropna(axis=1, how="all"))
    print("=" * 70)


# 2022 행유형 연속 후보 1건: 지역·원본행 키 기반 판정과 원본 문맥 진단
rowtype_decisions = pd.DataFrame(
    [
        {
            "지역": "서울",
            "후보_원본행": 28,
            "판정": "앞 행 병합",
            "병합대상_원본행": 24,
            "병합후_세부사업명": "학대피해아동 보호를 위한 안전망 구축",
            "판정근거": (
                "원본행 25~27은 페이지 제목·반복 머리글이고, 원본행 28은 "
                "앞 실제 데이터행 원본행 24의 세부사업명과 주요내용이 이어지는 행이다."
            ),
        }
    ]
)
decision_key_cols = ["지역", "후보_원본행"]
assert not rowtype_decisions.duplicated(decision_key_cols).any()

decision = rowtype_decisions.iloc[0]
candidate_mask = df_raw["지역"].eq(decision["지역"]) & df_raw["원본행"].eq(
    decision["후보_원본행"]
)
rowtype_candidates = df_raw.loc[candidate_mask].copy()
assert len(rowtype_candidates) == 1, (
    f"2022 행유형 연속 후보는 정확히 1건이어야 합니다: {len(rowtype_candidates)}건"
)
assert rowtype_candidates.iloc[0]["행유형"] == "구분행사업명 연속 후보"

candidate_idx = rowtype_candidates.index[0]
candidate_pos = df_raw.index.get_loc(candidate_idx)
region_rows = df_raw[df_raw["지역"].eq(decision["지역"])].sort_values("원본행")
actual_data_mask = region_rows["행유형"].eq("데이터행") & region_rows[
    "사업행구분"
].eq("세부사업")
previous_leaf = region_rows[
    actual_data_mask & region_rows["원본행"].lt(decision["후보_원본행"])
].tail(1)
next_leaf = region_rows[
    actual_data_mask & region_rows["원본행"].gt(decision["후보_원본행"])
].head(1)
assert len(previous_leaf) == 1 and len(next_leaf) == 1
assert int(previous_leaf.iloc[0]["원본행"]) == int(decision["병합대상_원본행"])

target_mask = df_raw["지역"].eq(decision["지역"]) & df_raw["원본행"].eq(
    decision["병합대상_원본행"]
)
merge_target = df_raw.loc[target_mask].copy()
assert len(merge_target) == 1, (
    f"병합 대상 원본행 24는 정확히 1건이어야 합니다: {len(merge_target)}건"
)

previous_pos = df_raw.index.get_loc(previous_leaf.index[0])
boundary_rows = df_raw.iloc[previous_pos + 1 : candidate_pos]

candidate_context = pd.concat(
    [
        previous_leaf.assign(진단위치="앞 세부사업행"),
        boundary_rows.assign(진단위치="페이지 경계행"),
        rowtype_candidates.assign(진단위치="연속 후보행"),
        next_leaf.assign(진단위치="뒤 세부사업행"),
    ]
)
context_cols = [
    "진단위치",
    "지역",
    "원본행",
    "행유형",
    "사업행구분",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
]
display(candidate_context[context_cols])

table1_start_row = int(previous_leaf.iloc[0]["원본행"])
table1_end_row = int(next_leaf.iloc[0]["원본행"])
table1_candidate_context = df_table1.iloc[table1_start_row - 1 : table1_end_row].copy()
table1_candidate_context.insert(
    0,
    "Table1_엑셀행",
    range(table1_start_row, table1_end_row + 1),
)
table1_candidate_context = table1_candidate_context.dropna(axis=1, how="all")
display(table1_candidate_context)

# 원본행 24의 예산·재정구분 값은 보존하고, 이름과 실제 주요내용만 병합한다.
preserved_target_cols = [
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "증감액",
    "비율",
    "사업분류재정구분",
]
target_values_before = merge_target.iloc[0][preserved_target_cols].copy()
target_name_before = str(merge_target.iloc[0]["세부사업명"])
target_content_before = merge_target.iloc[0]["주요내용"]
candidate_name = str(rowtype_candidates.iloc[0]["세부사업명"])
candidate_content = rowtype_candidates.iloc[0]["주요내용"]


def has_actual_text(value) -> bool:
    """결측·빈 문자열·대시가 아닌 실제 텍스트인지 판별한다."""
    return pd.notna(value) and str(value).strip() not in {"", "-"}


content_parts = []
if has_actual_text(target_content_before):
    content_parts.append(str(target_content_before).strip())
if has_actual_text(candidate_content):
    content_parts.append(str(candidate_content).strip())
merged_content = "\n".join(content_parts) if content_parts else pd.NA

df_raw.loc[target_mask, "세부사업명"] = decision["병합후_세부사업명"]
df_raw.loc[target_mask, "주요내용"] = merged_content
df_raw["후보판정_제외"] = False
df_raw.loc[candidate_mask, "후보판정_제외"] = True

merged_target = df_raw.loc[target_mask]
assert len(merged_target) == 1
assert merged_target.iloc[0]["세부사업명"] == "학대피해아동 보호를 위한 안전망 구축"
pd.testing.assert_series_equal(
    merged_target.iloc[0][preserved_target_cols],
    target_values_before,
    check_names=False,
)
assert "위한 안전망 구축" not in str(candidate_content)
assert df_raw.loc[candidate_mask, "후보판정_제외"].sum() == 1

rowtype_review = pd.DataFrame(
    [
        {
            "지역": decision["지역"],
            "후보_원본행": int(decision["후보_원본행"]),
            "앞_데이터행_원본행": int(previous_leaf.iloc[0]["원본행"]),
            "뒤_데이터행_원본행": int(next_leaf.iloc[0]["원본행"]),
            "후보_세부사업명": candidate_name,
            "판정": decision["판정"],
            "병합대상_원본행": int(decision["병합대상_원본행"]),
            "병합전_세부사업명": target_name_before,
            "병합후_세부사업명": merged_target.iloc[0]["세부사업명"],
            "판정근거": decision["판정근거"],
        }
    ]
)
assert len(rowtype_review) == 1
assert rowtype_review.iloc[0]["판정"] == "앞 행 병합"
assert rowtype_review.iloc[0]["병합대상_원본행"] == 24
rowtype_review.to_csv(
    REPORTS_DIR / f"{YEAR}_행유형_후보_검토.csv",
    index=False,
    encoding="utf-8-sig",
)
display(rowtype_review)

[광주광역시] Table 1 행 구간: 1968~2210
정확 명칭 일치: 0건 / 핵심 명칭 후보: 1건


,1,10,14,19,24
2195,4.인구구조 변화에 대한 적응(자체),158,110,48,43.6


[전라남도] Table 1 행 구간: 7112~7761
정확 명칭 일치: 0건 / 핵심 명칭 후보: 1건


,1,11,15,20,26
7585,3. 모두의 역량이 고루\n발휘되는 사회(자체),317359.5,117935.6,199423.9,169.1


-> 실제 원본 파일에도 없는 것 확인

-> 서울 원본행 28은 반복 머리글 원본행 25~27을 건너뛴 앞 실제 데이터행 원본행 24에 병합했다. 후보 행을 leaf에서 한 번 제외한 최종 수치는 leaf 8,129행, long 16,258행이다.


In [12]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf(세부사업)만 추출하되 병합 완료 후보 행은 정확히 한 번 제외한다.
df_leaf = df_labeled[
    df_labeled["사업행구분"].eq("세부사업") & ~df_labeled["후보판정_제외"]
].copy()
candidate_leaf_count = len(
    df_leaf[
        df_leaf["지역"].eq(decision["지역"])
        & df_leaf["원본행"].eq(decision["후보_원본행"])
    ]
)
merged_leaf = df_leaf[
    df_leaf["지역"].eq(decision["지역"])
    & df_leaf["원본행"].eq(decision["병합대상_원본행"])
]
assert candidate_leaf_count == 0
assert len(merged_leaf) == 1
assert merged_leaf.iloc[0]["세부사업명"] == "학대피해아동 보호를 위한 안전망 구축"
pd.testing.assert_series_equal(
    merged_leaf.iloc[0][preserved_target_cols],
    target_values_before,
    check_names=False,
)
assert len(df_leaf) == 8129, f"후보 판정 후 leaf는 8,129행이어야 합니다: {len(df_leaf)}행"
print("후보 판정 후 최종 leaf shape:", df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

(8130, 15)
['세부사업명', '사업분류재정구분', '2022년 예산', '2021년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '대분류', '중분류', '지역']


,세부사업명,사업분류재정구분,2022년 예산,2021년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역
2,저출생 인식개선 캠페인,공통,58.0,43,15,34.9,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",1,2,6,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
3,어린이 안전 영상정보 인프라 구축,공통,38039.0,33280,4759,14.3,지원대상 : 어린이\n지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,1,2,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
4,입양아동 가족지원,공통,4780.0,4120,660,16,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정\n지원내용 : 입양아동양육보조금, 심리치료비 등",1,2,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
5,국공립어린이집 등 보육서비스 수준제고,공통,330577.0,322975,7602,2.4,지원대상 : 국공립어린이집 등\n지원내용 : 보육교직원 인건비 지원,1,2,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
6,국공립어린이집 확충,공통,15523.0,40000,-24477,-61.2,국공립어린이집 이용률 46%달성,1,2,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울


In [13]:
mask_non_numeric = (
    to_numeric_budget(df_leaf[CURRENT_BUDGET_COL]).isna()
    & df_leaf[CURRENT_BUDGET_COL].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", CURRENT_BUDGET_COL]])

,지역,세부사업명,2022년 예산


In [14]:
# QA: 중분류_소계 행 예산 vs leaf 합계 대조 (17개 시도 x 중분류 단위)
leaf_sum = (
    df_leaf.groupby(["지역", "중분류"])[CURRENT_BUDGET_COL]
    .sum()
    .reset_index()
    .rename(columns={CURRENT_BUDGET_COL: "leaf_합계"})
)

# 원본 소계 행 값 (중분류_소계 행에서 직접 가져온 값)
subtotal = df_labeled[df_labeled["사업행구분"] == "중분류_소계"][
    ["지역", "대분류", "세부사업명", CURRENT_BUDGET_COL, "원본행"]
].rename(columns={"세부사업명": "중분류", CURRENT_BUDGET_COL: "원본_소계값"})

qa = subtotal.merge(leaf_sum, on=["지역", "중분류"], how="left")
qa["원시값_완전동일_결과"] = np.where(
    qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치"
)
qa["원시값_완전동일_결과"].value_counts()

결과
일치     118
불일치     16
Name: count, dtype: int64

In [15]:
qa.head()

,지역,대분류,중분류,원본_소계값,leaf_합계,결과
0,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),2991826.0,2991826.0,일치
1,서울,Ⅰ. 공통사업,2. 건강하고 능동적인\n고령사회(공통),3712331.0,3712331.0,일치
2,서울,Ⅰ. 공통사업,3. 모두의 역량이 고루\n발휘되는 사회(공통),519907.0,519907.0,일치
3,서울,Ⅰ. 공통사업,4.인구구조 변화에 대한 적응(공통),75381.0,75381.0,일치
4,서울,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),282248.0,282248.0,일치


In [16]:
# 부동소수점 표현 노이즈를 제거하기 위해 차이는 소수점 첫째 자리로 고정한다.
qa["차이"] = (qa["leaf_합계"] - qa["원본_소계값"]).round(1)

# 원시값 완전 동일과 소수점 첫째 자리 반올림 후 차이 0.0을 별도 컬럼으로 유지한다.
qa["소수점첫째자리_반올림후_차이0.0_결과"] = qa["차이"].eq(0.0).map(
    {True: "일치", False: "불일치"}
)

qa["검토구분"] = np.select(
    [
        qa["소수점첫째자리_반올림후_차이0.0_결과"].eq("일치"),
        qa["차이"].abs().le(0.5),
    ],
    ["일치", "미세 차이"],
    default="실질 검토",
)
qa_mismatch = qa[
    qa["소수점첫째자리_반올림후_차이0.0_결과"] == "불일치"
].copy()
qa_micro_mismatch = qa_mismatch[qa_mismatch["차이"].abs().le(0.5)].copy()
qa_material = qa_mismatch[qa_mismatch["차이"].abs().gt(0.5)].copy()
assert len(qa_mismatch) == len(qa_micro_mismatch) + len(qa_material)

exact_matches = int(qa["원시값_완전동일_결과"].eq("일치").sum())
rounded_matches = int(
    qa["소수점첫째자리_반올림후_차이0.0_결과"].eq("일치").sum()
)
assert (exact_matches, len(qa) - exact_matches) == (118, 16)
assert (rounded_matches, len(qa) - rounded_matches) == (119, 15)
assert (len(qa_micro_mismatch), len(qa_material)) == (12, 3)
print(f"원시값 완전 동일: 일치 {exact_matches}건 / 불일치 {len(qa) - exact_matches}건")
print(
    "소수점 첫째 자리 반올림 후 차이 0.0: "
    f"일치 {rounded_matches}건 / 불일치 {len(qa) - rounded_matches}건"
)
print(f"반올림 후 불일치 분류: 미세 차이 {len(qa_micro_mismatch)}건 / 실질 검토 {len(qa_material)}건")

검증 대상: 134건 / 불일치: 15건


In [17]:
display(
    qa_mismatch[
        [
            "지역",
            "대분류",
            "중분류",
            "원본행",
            "원본_소계값",
            "leaf_합계",
            "차이",
            "원시값_완전동일_결과",
            "소수점첫째자리_반올림후_차이0.0_결과",
            "검토구분",
        ]
    ]
)

,지역,대분류,중분류,원본_소계값,leaf_합계,차이
8,부산,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),1432683.0,1432683.4,0.4
12,부산,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),372665.7,372665.8,0.1
14,부산,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),78316.5,78316.6,0.1
15,부산,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한\n적응(자체),2641.9,2642.0,0.1
37,광주,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),12167.0,12166.5,-0.5
39,대전,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),574193.8,574193.7,-0.1
65,경기,Ⅰ. 공통사업,3. 모두의 역량이 고루\n발휘되는 사회(공통),348109.2,348109.3,0.1
71,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),464214.8,464214.9,0.1
106,전남,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),92786.3,92786.4,0.1
114,경북,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),402270.0,402097.7,-172.3


# 2022 QA 실질 불일치 원인 확인 - 경북·경남 원본 대조

`차이 = leaf_합계 - 원본_소계값`을 소수점 첫째 자리로 반올림한 뒤, `abs(차이) > 0.5`인 실질 검토 대상 3건을 진단한다.

- 경북 `1. 함께 일하고 함께 돌보는 사회(자체)`: -172.3
- 경남 `3. 모두의 역량이 고루 발휘되는 사회(공통)`: -1391.0
- 경남 `4.인구구조 변화에 대한 적응(자체)`: +52.0

`원본행`은 `Table 1`의 1-indexed 엑셀 행 번호이므로, 소계 위치와 원본행 간격을 함께 출력해 입력자료 이상과 파싱 문제를 구분한다.


In [ ]:
expected_material = {
    ("경북", -172.3),
    ("경남", -1391.0),
    ("경남", 52.0),
}
actual_material = set(zip(qa_material["지역"], qa_material["차이"]))
assert actual_material == expected_material, actual_material

display(
    qa_material[
        ["지역", "대분류", "중분류", "원본행", "원본_소계값", "leaf_합계", "차이"]
    ]
)

In [ ]:
def show_labeled_subtotal_context(sido: str, medium_category: str, window: int = 2):
    """정리본_자동에서 대상 중분류 소계 앞뒤 문맥을 원본행 순서로 표시한다."""
    region_df = df_labeled[df_labeled["지역"].eq(sido)].sort_values("원본행")
    subtotal_rows = region_df[
        region_df["사업행구분"].eq("중분류_소계")
        & region_df["세부사업명"].eq(medium_category)
    ]
    assert len(subtotal_rows) == 1, (sido, medium_category, len(subtotal_rows))
    center_pos = region_df.index.get_loc(subtotal_rows.index[0])
    view = region_df.iloc[max(0, center_pos - window) : center_pos + window + 1]
    print(f"--- {sido} / {medium_category} / 정리본_자동 소계 문맥 ---")
    display(
        view[
            ["원본행", "세부사업명", "사업행구분", "대분류", "중분류", CURRENT_BUDGET_COL]
        ]
    )


for target in qa_material.itertuples(index=False):
    show_labeled_subtotal_context(target.지역, target.중분류)

In [ ]:
gap_diagnostics = []
for target in qa_material.itertuples(index=False):
    category_rows = df_labeled[
        df_labeled["지역"].eq(target.지역)
        & df_labeled["중분류"].eq(target.중분류)
        & df_labeled["사업행구분"].isin(["중분류_소계", "세부사업"])
    ].sort_values("원본행")
    gaps = category_rows[["원본행", "세부사업명", "사업행구분"]].copy()
    gaps["이전_원본행"] = gaps["원본행"].shift()
    gaps["이전_세부사업명"] = gaps["세부사업명"].shift()
    gaps["원본행_간격"] = gaps["원본행"] - gaps["이전_원본행"]
    gaps = gaps[gaps["원본행_간격"].gt(1)].copy()
    gaps.insert(0, "중분류", target.중분류)
    gaps.insert(0, "지역", target.지역)
    gap_diagnostics.append(gaps)

qa_source_gaps = pd.concat(gap_diagnostics, ignore_index=True)
display(qa_source_gaps)

In [ ]:
def show_table1_around(center_excel_row: int, window: int, label: str):
    """Table 1 원본 시트에서 특정 엑셀 행(1-indexed) 주변의 비어 있지 않은 열을 표시한다."""
    start = max(1, center_excel_row - window)
    end = min(len(df_table1), center_excel_row + window)
    print(f"--- {label} (Table 1 엑셀행 {start}~{end}) ---")
    view = df_table1.iloc[start - 1 : end].copy()
    view.insert(0, "Table1_엑셀행", range(start, end + 1))
    view = view.dropna(axis=1, how="all")
    display(view)


region_name_map = {"경북": "경상북도", "경남": "경상남도"}


def show_table1_subtotal_context(sido: str, medium_category: str, window: int = 3):
    """해당 시도 원문 구간에서 중분류 소계를 찾아 Table 1 문맥을 표시한다."""
    full_region_name = region_name_map[sido]
    start_pos = next(i for i, (_, name) in enumerate(region_starts) if name == full_region_name)
    start_row = region_starts[start_pos][0]
    end_row = (
        region_starts[start_pos + 1][0] if start_pos + 1 < len(region_starts) else len(df_table1)
    )
    normalized_title = re.sub(r"\s+", "", medium_category)
    region_text = table1_row_text.iloc[start_row:end_row]
    matches = region_text[region_text.str.contains(re.escape(normalized_title), regex=True)]
    assert len(matches) == 1, (sido, medium_category, list(matches.index + 1))
    show_table1_around(
        int(matches.index[0]) + 1,
        window,
        f"{sido} / {medium_category}",
    )


for target in qa_material.itertuples(index=False):
    show_table1_subtotal_context(target.지역, target.중분류)

print("원본행 간격 후보는 위 qa_source_gaps와 Table 1 원문을 함께 대조해 원인을 판정한다.")

# 2022년 증감액 부호 검증

2022년 원본 `증감액`의 부호가 `2022년 예산 - 2021년 예산`과 일치하는지 전국 17개 시도에서 확인한다.

당해·전년도 예산과 원본 증감액은 모두 상단의 `to_numeric_budget`로 변환하며, 최종 증감액·증감율은 당해·전년도 예산으로 직접 재계산한다.


In [ ]:
# 상단의 공통 숫자 변환 함수를 당해·전년도 예산과 원본 증감액에 재사용한다.
df_raw[f"{YEAR}년 예산_num"] = to_numeric_budget(df_raw[CURRENT_BUDGET_COL])
df_raw[f"{PREVIOUS_YEAR}년 예산_num"] = to_numeric_budget(df_raw[PREVIOUS_BUDGET_COL])
df_raw["증감액_num"] = to_numeric_budget(df_raw["증감액"])
df_raw["계산된_증감액"] = df_raw[f"{YEAR}년 예산_num"] - df_raw[f"{PREVIOUS_YEAR}년 예산_num"]

# 실제로 예산이 감소한 행(계산된 증감액 < 0)만 골라서, 증감액 컬럼 부호를 대조
감소행 = df_raw[df_raw["계산된_증감액"] < -0.5].copy()
감소행["부호_소실"] = 감소행["증감액_num"] > 0.5

print("예산 감소 행 수:", len(감소행))
print("그중 증감액이 양수로 찍힌(부호 소실) 행 수:", 감소행["부호_소실"].sum())

In [ ]:
# 지역별로 부호 소실 비율 확인
지역별_부호소실_비율 = (
    감소행.groupby("지역")["부호_소실"].mean().mul(100).round(1).sort_values(ascending=False)
)
지역별_부호소실_비율

# 증감액 / 비율 직접 재계산

- 위에서 확인했듯이 원본에서는 시도별로 부호나 값 자체가 틀린 경우가 있어 신뢰할 수 없음.
- 따라서 직접 재계산


In [ ]:
# 상단의 공통 숫자 변환 함수를 leaf 예산 재계산에도 재사용한다.
current_num_col = f"{YEAR}년_예산_num"
previous_num_col = f"{PREVIOUS_YEAR}년_예산_num"
df_leaf[previous_num_col] = to_numeric_budget(df_leaf[PREVIOUS_BUDGET_COL])
df_leaf[current_num_col] = to_numeric_budget(df_leaf[CURRENT_BUDGET_COL])
df_leaf["증감액_재계산"] = df_leaf[current_num_col] - df_leaf[previous_num_col]
df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"] / df_leaf[previous_num_col].replace(0, np.nan) * 100
).round(1)

df_leaf[["세부사업명", current_num_col, previous_num_col, "증감액_재계산", "증감율_재계산"]].head(
    10
)

# 최종 스키마


In [ ]:
# 텍스트 정리
PUA_PATTERN = re.compile(r"[\uE000-\uF8FF]")


def clean_text(series: pd.Series) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리"""

    def _clean(x):
        if pd.isna(x):
            return x
        x = PUA_PATTERN.sub("", str(x))
        return re.sub(r"\s+", " ", x).strip()

    return series.apply(_clean)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"])
df_leaf["대분류"] = clean_text(df_leaf["대분류"])
df_leaf["중분류"] = clean_text(df_leaf["중분류"])

df_leaf.head()

In [ ]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            current_num_col: "당해예산",
            previous_num_col: "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=YEAR)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

assert len(df_leaf_final) == 8129
print(df_leaf_final.shape)
display(df_leaf_final.head())

In [ ]:
for sido in df_leaf_final["지역"].unique():
    sido_labeled = df_labeled[df_labeled["지역"] == sido]

    sido_dir = get_sido_dir(sido)

    sido_labeled.to_csv(
        sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv", index=False, encoding="utf-8-sig"
    )

# QA 결과는 전체 17개 시도 한 파일로 저장한다(원시값 완전 동일/첫째 자리 반올림 결과 포함).
qa.to_csv(REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

# 세부사업명 오탈자 / 중복 탐지

- 오탈자를 자동으로 고치지 않고, 사람이 검토할 후보만 찾는다.
- 같은 지역 / 중분류 안에서 완전히 같지는 않지만 유사도가 높은 세부사업명 쌍을 `rapidfuzz`로 찾는다.


In [ ]:
from itertools import combinations
from rapidfuzz import fuzz

In [ ]:
def normalize_for_match(name: str) -> str:
    """clean_text로 정제된 세부사업명에서, 비교 목적으로 공백, 문장부호 마저 제거"""
    return re.sub(r"[\s,-./\-/()]", "", name)


def find_near_duplicates(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    """같은 지역 중분류 안에서 완전 일치는 아니지만 유사도가 높은 세부사업명과 쌍을 찾는다."""
    candidates = []
    for (sido, midium_cat), group in df.groupby(["지역", "중분류"]):
        rows = list(
            group[["세부사업명", "당해예산", "주요내용"]].itertuples(index=False, name=None)
        )
        for (name_a, budget_a, content_a), (name_b, budget_b, content_b) in combinations(rows, 2):
            if name_a == name_b:
                continue
            if normalize_for_match(name_a) == normalize_for_match(name_b):
                continue  # 공백/문장부호 차이 -> 검수 대상 x
            score = fuzz.ratio(name_a, name_b)
            if score >= threshold:
                candidates.append(
                    {
                        "지역": sido,
                        "중분류": midium_cat,
                        "세부사업명1": name_a,
                        "당해예산1": budget_a,
                        "주요내용1": content_a,
                        "세부사업명2": name_b,
                        "당해예산2": budget_b,
                        "주요내용2": content_b,
                        "유사도": score,
                    }
                )
    return pd.DataFrame(candidates).sort_values(
        "유사도", ascending=False, kind="stable"
    )


near_dup = find_near_duplicates(df_leaf_final)
print(len(near_dup), "건 발견")
display(near_dup.head(10))

In [ ]:
# 당해예산이 완전히 같은 쌍은 진짜 같은 사업일 가능성이 높은 후보
near_dup_same_budget = near_dup[near_dup["당해예산1"] == near_dup["당해예산2"]]

print(len(near_dup_same_budget), "건 (전체", len(near_dup), "건 중)")
display(near_dup_same_budget)

In [ ]:
# 당해예산 0.0이 우연일치인지 확인
print("당해예산 = 0 으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] == 0).sum())
print("0이 아닌 값으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] != 0).sum())

near_dup_confident = near_dup_same_budget[near_dup_same_budget["당해예산1"] != 0]
display(near_dup_confident)

# 기존 리포트의 스키마·컬럼 순서·정렬(고신뢰 우선, 유사도 내림차순)을 유지한다.
near_dup_report = (
    near_dup.assign(
        유사도=near_dup["유사도"].round(1),
        예산일치_신뢰도높음=(
            near_dup["당해예산1"].eq(near_dup["당해예산2"])
            & near_dup["당해예산1"].ne(0)
        ),
    )
    .rename(
        columns={
            "세부사업명1": "세부사업명_A",
            "당해예산1": "예산_A",
            "세부사업명2": "세부사업명_B",
            "당해예산2": "예산_B",
        }
    )
    [
        [
            "지역",
            "중분류",
            "유사도",
            "세부사업명_A",
            "예산_A",
            "세부사업명_B",
            "예산_B",
            "예산일치_신뢰도높음",
        ]
    ]
    .sort_values(
        ["예산일치_신뢰도높음", "유사도"],
        ascending=[False, False],
        kind="stable",
    )
    .reset_index(drop=True)
)
near_dup_report.to_csv(
    REPORTS_DIR / f"{YEAR}_세부사업명_오탈자중복_후보.csv",
    index=False,
    encoding="utf-8-sig",
)
print(
    "near duplicate 저장:",
    len(near_dup_report),
    "건 / 고신뢰",
    int(near_dup_report["예산일치_신뢰도높음"].sum()),
    "건",
)

# 주요내용 구조 패턴 검사

- 원자화를 위해 '지원대상:..', '지원내용: ..' 이 모든 행에 포함되어있는지 확인한다.


In [ ]:
def dedup_label(text: str) -> str:
    """지원대상 : 지원대상 : ...  처럼 같은 라벨이 연속으로 중복된 걸 하나로 정리"""
    if pd.isna(text):
        return text
    text = re.sub(
        r"\(\s*(지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\s*\)",
        r"\1 : ",
        text,
    )
    text = re.sub(r"(지원대상\s*[:：]\s*)+", "지원대상 : ", text)
    text = re.sub(r"(지원내용\s*[:：]\s*)+", "지원내용 : ", text)
    text = re.sub(r"(사업대상\s*[:：]\s*)+", "사업대상 : ", text)
    text = re.sub(r"(사업내용\s*[:：]\s*)+", "사업내용 : ", text)
    return text


df_leaf_final["주요내용"] = df_leaf_final["주요내용"].apply(dedup_label)

In [ ]:
support_pattern = re.compile(r"^지원대상\s*[:：]\s*(.*?)\s*지원내용\s*[:：]\s*(.*)$", re.DOTALL)


def check_pattern(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern.match(text) else "불일치"


df_leaf_final["주요내용_패턴"] = df_leaf_final["주요내용"].apply(check_pattern)

print(df_leaf_final["주요내용_패턴"].value_counts())
print(df_leaf_final["주요내용_패턴"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# 범위 넓혀보기
support_pattern_broad = re.compile(
    r"^(지원대상|사업대상|대상)\s*[:：]?\s*(.*?)\s*(지원내용|사업내용|내용)\s*[:：]?\s*(.*)$",
    re.DOTALL,
)


def check_pattern_broad(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern_broad.match(text) else "불일치"


df_leaf_final["주요내용_패턴_확장"] = df_leaf_final["주요내용"].apply(check_pattern_broad)

print(df_leaf_final["주요내용_패턴_확장"].value_counts())
print(df_leaf_final["주요내용_패턴_확장"].value_counts(normalize=True).mul(100).round(1))

# 여전히 안걸리는 샘플 확인
display(
    df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"][
        ["지역", "세부사업명", "주요내용"]
    ].head(20)
)

In [ ]:
def extract_via_regex(text: str) -> dict:
    """패턴에 걸리면 그대로 분리, 안 걸리면 None"""
    if pd.isna(text):
        return {"지원대상": None, "지원내용": None}
    m = support_pattern_broad.match(text)
    if m:
        return {"지원대상": m.group(2).strip(), "지원내용": m.group(4).strip()}
    return {"지원대상": None, "지원내용": None}


regex_extracted = pd.json_normalize(df_leaf_final["주요내용"].apply(extract_via_regex))
df_leaf_final["지원대상"] = regex_extracted["지원대상"]
df_leaf_final["지원내용_상세"] = regex_extracted["지원내용"]

In [ ]:
df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"].head()

# 주요내용 LLM 정제 (업스테이지 Solar Pro 3, 구조화 출력)

- LLM은 `주요내용_정제`(오탈자·이상한 공백만 보존형으로 교정) 생성만 담당한다. 요약/재구성 없음, 숫자·고유명사 변경 없음.
- `지원대상`/`지원내용_상세`는 이미 검증된 정규식(`extract_via_regex`) 결과를 그대로 쓴다 — 정규식으로 걸러지는 데 LLM을 또 쓰는 건 실익이 없고 과잉분리 위험만 늘린다.
- `response_format`(json_schema)으로 API 단에서 출력 구조를 강제, 실패 시 1회 재시도 후에도 실패하면 원문을 그대로 사용(정제문장 결측 방지).
- 정제 전후 숫자(금액·비율·연령 등) 보존 여부를 자동 검증해, 달라진 행은 검토 대상으로 표시한다.


In [ ]:
import os
import json
import yaml
from openai import OpenAI
from tqdm import tqdm

tqdm.pandas()

with open("../configs/llm_extraction.yaml") as f:
    llm_cfg = yaml.safe_load(f)

client = OpenAI(
    api_key=os.environ["UPSTAGE_API_KEY"],
    base_url=llm_cfg["upstage"]["base_url"],
)

RESPONSE_FORMAT = {"type": "json_schema", "json_schema": llm_cfg["response_schema"]}


def call_llm_once(name: str, content: str) -> str | None:
    """API 1회 호출. 파싱 실패하면 None 반환"""
    prompt = llm_cfg["prompt"]["template"].format(name=name, content=content)
    response = client.chat.completions.create(
        model=llm_cfg["upstage"]["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=llm_cfg["upstage"]["temperature"],
        response_format=RESPONSE_FORMAT,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)["정제문장"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


def clean_sentence(name: str, content: str) -> str:
    """주요내용을 LLM으로 정제. 결측이면 결측 유지, 실패 시(1회 재시도 포함) 원문 그대로 사용"""
    if pd.isna(content):
        return None
    for attempt in range(2):  # 최초 시도 + 1회 재시도
        try:
            result = call_llm_once(name, content)
        except Exception as e:
            print(f"API 호출 실패({attempt + 1}회차): {name} -> {e}")
            result = None
        if result is not None:
            return result
    print(f"정제 실패, 원문 유지: {name}")
    return content


def extract_numbers(text) -> list:
    """숫자(금액·비율·연령 등) 시퀀스만 뽑아 리스트로 반환 - 원문/정제문장 보존 검증용"""
    if pd.isna(text):
        return []
    return re.findall(r"\d+", text)


def numbers_preserved(original, cleaned) -> bool:
    """정제 과정에서 숫자가 그대로 보존됐는지 확인 (다르면 검토 대상)"""
    return extract_numbers(original) == extract_numbers(cleaned)

# 먼저 소량 샘플로 속도·품질 확인 (전체 실행 전 확인용)

```python
sample = df_leaf_final.sample(
    llm_cfg["sample"]["size"], random_state=llm_cfg["sample"]["random_state"]
).copy()

sample["주요내용_정제"] = sample.progress_apply(
    lambda row: clean_sentence(row["세부사업명"], row["주요내용"]), axis=1
)
sample["숫자보존"] = sample.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~sample["숫자보존"]).sum())
display(
    sample[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세", "숫자보존"]]
)
```


샘플 결과가 괜찮으면 아래 셀로 전체 실행 (7천여 건, 시간 오래 걸릴 수 있음)


In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"

PUA_LOW, PUA_HIGH = chr(0xE000), chr(0xF8FF)
pua_re = re.compile(f"[{PUA_LOW}-{PUA_HIGH}]")
paren_label_re = re.compile(
    r"\((지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\)"
)


def needs_llm_rerun(text):
    if pd.isna(text):
        return False
    return bool(pua_re.search(text)) or bool(paren_label_re.search(text))


# 체크포인트 파일을 직접 읽어서 판별한다(df_leaf_final 병합을 기다릴 필요 없음).
# 이렇게 해야 LLM 처리 셀보다 앞에 둘 수 있고, 한 번의 순차 실행(Run All)만으로
# 재실행 대상까지 자동으로 포함되어 처리된다 (CodeRabbit 리뷰 반영).
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    affected_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
    affected_idx = checkpoint_df.index[affected_mask]

    if len(affected_idx) > 0:
        checkpoint_df = checkpoint_df.drop(index=affected_idx)
        checkpoint_df.to_csv(CHECKPOINT_PATH)

    print(
        "재실행 대상(체크포인트에서 제거):",
        len(affected_idx),
        "건 -> 남은 행수:",
        len(checkpoint_df),
    )
else:
    print("체크포인트 파일 없음 - 전체 신규 실행")

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
CHUNK_SIZE = 200

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    done_index = set(checkpoint_df.index)
    print(f"체크포인트 발견: {len(done_index)}건 이미 처리됨, 이어서 진행")
else:
    checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
    done_index = set()

targets = df_leaf_final.index.difference(done_index)
results = {}

for i, idx in enumerate(tqdm(targets), start=1):
    row = df_leaf_final.loc[idx]
    results[idx] = clean_sentence(row["세부사업명"], row["주요내용"])

    if i % CHUNK_SIZE == 0:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH)
        results = {}

# 남은 것 마저 저장
if results:
    partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
    checkpoint_df = pd.concat([checkpoint_df, partial])
    checkpoint_df.to_csv(CHECKPOINT_PATH)

df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~df_leaf_final["숫자보존"]).sum())
display(
    df_leaf_final[~df_leaf_final["숫자보존"]][
        ["지역", "세부사업명", "주요내용", "주요내용_정제"]
    ].head(20)
)

df_leaf_final[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세"]].head(20)

In [ ]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]

missing_cols = [c for c in output_cols if c not in df_leaf_final.columns]

if missing_cols:
    raise KeyError(f"df_leaf_final에 없는 칼럼: {missing_cols}")

for sido_name, group in df_leaf_final.groupby("지역"):
    out_path = get_sido_dir(sido_name) / f"{YEAR}_{sido_name}_세부사업_정제.csv"
    group[output_cols].to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")


print("\n숫자보존 불일치 총 건수:", (~df_leaf_final["숫자보존"]).sum())

In [ ]:
df_leaf_final.columns.tolist()

In [ ]:
id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]

df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

# 전년도예산 행은 실제 그 돈이 집행된 연도로 맞춰서 -1
previous_mask = df_long["예산구분"] == "전년도예산"
df_long.loc[previous_mask, "연도"] -= 1
# 증감액/증감율은 "당해 대비 전년" 개념이라 전년도 행에는 그대로 복제되면 안 됨 (CodeRabbit 지적)
df_long.loc[previous_mask, ["증감액", "증감율"]] = None

df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

assert len(df_leaf_final) == 8129
assert len(df_long) == 16258
print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf_final), "행 x 2)")
df_long.head(10)

In [ ]:
# 시도별로 저장
for sido_name, group in df_long.groupby("지역"):
    out_path = get_sido_dir(sido_name) / f"{YEAR}_{sido_name}_세부사업_정제_long.csv"
    group.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")